## Once and for all

I need to know what are the minimal models that are enough for CIFAR10.

In [24]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import tqdm
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import MNIST, CIFAR10
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import mutual_info_score
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots
from IPython.display import clear_output
from collections import defaultdict
from itertools import islice
import random
import time
from pathlib import Path
import math

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device

def randomseed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
def accuracy(model, data):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data:
            outputs = model(images.to(device))
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels.to(device)).sum().item()
    return correct / total

train_dataset = CIFAR10(root='.', train=True, download=True, transform=torchvision.transforms.ToTensor())
test_dataset = CIFAR10(root='.', train=False, download=True, transform=torchvision.transforms.ToTensor())

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True, pin_memory=True, num_workers=4)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False, pin_memory=True, num_workers=4)

Files already downloaded and verified
Files already downloaded and verified


In [25]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(3072, 64, bias=False)
        self.fc2 = nn.Linear(64, 64, bias=False)
        self.fc3 = nn.Linear(64, 10, bias=False)

    def forward(self, x):
        x = x.view(-1, 3072)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [75]:
class bigger_MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc0 = nn.Linear(3072, 256, bias=False)
        self.fc1 = nn.Linear(256, 64, bias=False)
        self.fc2 = nn.Linear(64, 64, bias=False)
        self.fc3 = nn.Linear(64, 10, bias=False)

    def forward(self, x):
        x = x.view(-1, 3072)
        x = torch.relu(self.fc0(x))
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [71]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(16 * 16 * 16, 64, bias=False)
        self.fc2 = nn.Linear(64, 64, bias=False)
        self.fc3 = nn.Linear(64, 10, bias=False)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = x.view(-1, 16 * 16 * 16)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [61]:
# get the whole test set in one big tensor for faster evaluation

test_images = torch.cat([img for img, _ in test_loader], dim=0).to(device)
test_labels = torch.cat([lab for _, lab in test_loader], dim=0).to(device)

def fast_acc(model):
    global test_images, test_labels
    model.eval()
    with torch.no_grad():
        outputs = model(test_images)
        _, predicted = torch.max(outputs.data, 1)
        correct = (predicted == test_labels).sum().item()
        return correct / test_labels.size(0)

In [83]:
# model = MLP()
model = CNN()
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.7)
epochs = 10

In [84]:
fast_acc(model)

0.0973

In [85]:
for epoch in range(epochs):

    model.train()

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

    lr_scheduler.step()
    acc = fast_acc(model)
    print(f'Epoch {epoch+1}/{epochs}, Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}, Accuracy: {acc:.4f}')

Epoch 1/10, Batch 781/782, Loss: 1.2435, Accuracy: 0.4901
Epoch 2/10, Batch 781/782, Loss: 1.6384, Accuracy: 0.5465
Epoch 3/10, Batch 781/782, Loss: 1.6952, Accuracy: 0.6109
Epoch 4/10, Batch 781/782, Loss: 0.7526, Accuracy: 0.6355
Epoch 5/10, Batch 781/782, Loss: 1.0070, Accuracy: 0.6491
Epoch 6/10, Batch 781/782, Loss: 0.5265, Accuracy: 0.6522
Epoch 7/10, Batch 781/782, Loss: 1.3242, Accuracy: 0.6569
Epoch 8/10, Batch 781/782, Loss: 1.0312, Accuracy: 0.6505
Epoch 9/10, Batch 781/782, Loss: 0.5551, Accuracy: 0.6611
Epoch 10/10, Batch 781/782, Loss: 0.9969, Accuracy: 0.6645


So CIFAR10 is much harder than MNIST. I either need residuals or big CNNs to get to something like 90% accuracy. It's better to feel okay with working with 50% accuracy here (the PyTorch blog does this as well).